# Phase 2C — Inferential Statistics & Hypothesis Testing

**Theory covered:** p-values, confidence intervals, t-tests, chi-square test, ANOVA, correlation.

**Tools used:** `scipy.stats`, `statsmodels`, `numpy`, `matplotlib`

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---
## 1. The Hypothesis Testing Framework

1. State **H₀** (null hypothesis) and **H₁** (alternative hypothesis)
2. Choose significance level **α** (usually 0.05)
3. Compute the **test statistic** from data
4. Find the **p-value** — probability of observing this result (or more extreme) if H₀ is true
5. **Decision:** if p-value < α, reject H₀

**Important:** p-value is NOT the probability H₀ is true. It's the probability of your data given H₀.

---
## 2. One-Sample t-Test

**Question:** Is the sample mean significantly different from a known/hypothesized value?

**Use case:** Is the average delivery time of our app different from the industry standard of 35 minutes?

In [ ]:
# Delivery times (minutes) from a sample of 40 orders
delivery_times = np.array(
    [
        38,
        32,
        41,
        35,
        29,
        44,
        37,
        33,
        40,
        36,
        31,
        45,
        39,
        34,
        28,
        43,
        38,
        36,
        41,
        30,
        37,
        35,
        42,
        33,
        39,
        40,
        34,
        36,
        38,
        31,
        29,
        44,
        37,
        40,
        35,
        33,
        41,
        36,
        38,
        34,
    ]
)

hypothesized_mean = 35  # industry standard

t_stat, p_value = stats.ttest_1samp(delivery_times, popmean=hypothesized_mean)

print(f"Sample mean   : {delivery_times.mean():.2f} minutes")
print(f"Hypothesized  : {hypothesized_mean} minutes")
print(f"t-statistic   : {t_stat:.4f}")
print(f"p-value       : {p_value:.4f}")
print()
alpha = 0.05
if p_value < alpha:
    print(
        f"REJECT H₀ — delivery time is significantly different from {hypothesized_mean} min (p={p_value:.4f})"
    )
else:
    print(
        f"FAIL TO REJECT H₀ — no significant difference from {hypothesized_mean} min (p={p_value:.4f})"
    )

---
## 3. Two-Sample Independent t-Test

**Question:** Are the means of two independent groups significantly different?

**Use case:** Does Model A produce different predictions than Model B? Do users in Group A engage more than Group B (A/B testing)?

In [ ]:
# A/B test: click-through time (seconds) for two website designs
group_a = np.random.normal(loc=8.5, scale=2.0, size=50)  # Design A
group_b = np.random.normal(loc=9.2, scale=2.0, size=50)  # Design B

t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=True)

print(f"Group A mean: {group_a.mean():.3f}s")
print(f"Group B mean: {group_b.mean():.3f}s")
print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_value:.4f}")

# Effect size — Cohen's d
pooled_std = np.sqrt((group_a.std() ** 2 + group_b.std() ** 2) / 2)
cohens_d = (group_b.mean() - group_a.mean()) / pooled_std
print(f"Cohen's d   : {cohens_d:.4f}  (small: 0.2, medium: 0.5, large: 0.8)")

alpha = 0.05
decision = "REJECT H₀" if p_value < alpha else "FAIL TO REJECT H₀"
print(f"\nDecision: {decision} (α={alpha})")

In [ ]:
# Visualize the two groups
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(group_a, bins=20, alpha=0.6, density=True, color="steelblue", label="Group A")
ax.hist(group_b, bins=20, alpha=0.6, density=True, color="coral", label="Group B")
ax.axvline(group_a.mean(), color="steelblue", linestyle="--", lw=2)
ax.axvline(group_b.mean(), color="coral", linestyle="--", lw=2)
ax.set_title(f"A/B Test — Two-Sample t-Test (p={p_value:.3f})")
ax.set_xlabel("Time on Page (seconds)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Confidence Intervals

A 95% confidence interval means: if we repeated the experiment 100 times, the true population mean would fall inside the interval 95 times.

**Formula:** `CI = x̄ ± t*(s/√n)`

In [ ]:
# 95% confidence interval for delivery times
n = len(delivery_times)
x_bar = delivery_times.mean()
se = stats.sem(delivery_times)  # standard error of the mean

ci_low, ci_high = stats.t.interval(confidence=0.95, df=n - 1, loc=x_bar, scale=se)

print(f"Sample mean    : {x_bar:.2f} minutes")
print(f"Standard error : {se:.4f}")
print(f"95% CI         : ({ci_low:.2f}, {ci_high:.2f}) minutes")
print()
print(f"Interpretation: We are 95% confident the true mean delivery time is between")
print(f"{ci_low:.2f} and {ci_high:.2f} minutes.")

---
## 5. Chi-Square Test for Independence

**Question:** Is there a significant association between two categorical variables?

**Use case:** Is customer churn independent of subscription tier?

In [ ]:
# Contingency table: Subscription Tier vs Churn
#                  Churned   Retained
# Basic               45        155
# Premium             15        285

observed = np.array(
    [
        [45, 155],  # Basic tier
        [15, 285],  # Premium tier
    ]
)

chi2, p_value, dof, expected = stats.chi2_contingency(observed)

print("Observed frequencies:")
print(f"  Basic   — Churned: {observed[0, 0]}, Retained: {observed[0, 1]}")
print(f"  Premium — Churned: {observed[1, 0]}, Retained: {observed[1, 1]}")
print()
print(f"Chi-square statistic : {chi2:.4f}")
print(f"Degrees of freedom   : {dof}")
print(f"p-value              : {p_value:.6f}")
print()
print("Expected frequencies (if independent):")
print(f"  Basic   — Churned: {expected[0, 0]:.1f}, Retained: {expected[0, 1]:.1f}")
print(f"  Premium — Churned: {expected[1, 0]:.1f}, Retained: {expected[1, 1]:.1f}")
print()

if p_value < 0.05:
    print("REJECT H₀ — Churn is NOT independent of subscription tier")
else:
    print("FAIL TO REJECT H₀ — No significant association found")

---
## 6. One-Way ANOVA

**Question:** Are the means of THREE or more independent groups significantly different?

**Use case:** Does ad format (video, banner, text) affect user engagement time?

In [ ]:
# Engagement time (seconds) by ad format
video_ad = np.random.normal(loc=15, scale=3, size=30)
banner_ad = np.random.normal(loc=12, scale=3, size=30)
text_ad = np.random.normal(loc=10, scale=3, size=30)

f_stat, p_value = stats.f_oneway(video_ad, banner_ad, text_ad)

print(f"Video  mean: {video_ad.mean():.2f}s")
print(f"Banner mean: {banner_ad.mean():.2f}s")
print(f"Text   mean: {text_ad.mean():.2f}s")
print()
print(f"F-statistic : {f_stat:.4f}")
print(f"p-value     : {p_value:.6f}")
print()

if p_value < 0.05:
    print("REJECT H₀ — At least one ad format has significantly different engagement")
    print("Next step: Run post-hoc tests (Tukey HSD) to find which pairs differ")
else:
    print("FAIL TO REJECT H₀ — No significant difference between ad formats")

---
## 7. Pearson Correlation

Measures the **linear relationship** between two continuous variables. Range: -1 to +1.

In [ ]:
# Study hours vs exam scores
study_hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
exam_scores = np.array([50, 55, 60, 65, 70, 72, 78, 85, 88, 95])

r, p_value = stats.pearsonr(study_hours, exam_scores)

print(f"Pearson r : {r:.4f}")
print(f"p-value   : {p_value:.6f}")

strength = "weak" if abs(r) < 0.3 else "moderate" if abs(r) < 0.7 else "strong"
direction = "positive" if r > 0 else "negative"
print(f"Interpretation: {strength} {direction} correlation")

# Spearman correlation (rank-based, handles non-linear monotonic relationships)
rho, p_spearman = stats.spearmanr(study_hours, exam_scores)
print(f"\nSpearman ρ: {rho:.4f} (use this when data is not normally distributed)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(study_hours, exam_scores, color="steelblue", s=80, zorder=3)
m, b = np.polyfit(study_hours, exam_scores, 1)
ax.plot(study_hours, m * study_hours + b, "r-", label=f"Trend line (r={r:.3f})")
ax.set_title("Study Hours vs Exam Score")
ax.set_xlabel("Study Hours")
ax.set_ylabel("Score")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary — Which Test to Use?

| Situation | Test | `scipy.stats` function |
|-----------|------|------------------------|
| Sample mean vs known value | One-sample t-test | `ttest_1samp` |
| Two independent group means | Two-sample t-test | `ttest_ind` |
| Two paired group means | Paired t-test | `ttest_rel` |
| Three+ group means | One-way ANOVA | `f_oneway` |
| Two categorical variables | Chi-square test | `chi2_contingency` |
| Linear relationship | Pearson correlation | `pearsonr` |
| Monotonic relationship | Spearman correlation | `spearmanr` |